In [1]:
#cd /kaggle/working/

In [2]:
!git clone https://github.com/vinhVVN/Realized-Volatility-Prediction.git

Cloning into 'Realized-Volatility-Prediction'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 121 (delta 46), reused 102 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 76.83 KiB | 1.67 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [3]:
# import shutil

# # Thay đổi đường dẫn đến thư mục bạn muốn xóa
# folder_path = '/kaggle/working/Realized-Volatility-Prediction'

# try:
#     shutil.rmtree(folder_path)
#     print(f"Đã xóa thành công thư mục: {folder_path}")
# except FileNotFoundError:
#     print("Thư mục không tồn tại.")
# except Exception as e:
#     print(f"Có lỗi xảy ra: {e}")

In [4]:
import os
import sys
import gc
import numpy as np
import pandas as pd
import warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# 1. Định vị thư mục gốc của dự án trên Kaggle
PROJECT_ROOT = '/kaggle/working/Realized-Volatility-Prediction'
if os.path.exists(PROJECT_ROOT):
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"🚀 Môi trường Deep Learning đã sẵn sàng tại: {os.getcwd()}")
print(f"🖥️ Thiết bị tính toán hiện tại: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU (30GB RAM)'}")

# Cấu hình thiết bị chạy PyTorch (Kaggle CPU mode cực kỳ mạnh về RAM)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

🚀 Môi trường Deep Learning đã sẵn sàng tại: /kaggle/working/Realized-Volatility-Prediction
🖥️ Thiết bị tính toán hiện tại: GPU (CUDA)


In [5]:
from src.training.cv_splitter import TimeSeriesCVSplitter
from src.training.metrics import rmspe

# Import các lớp mô hình PyTorch từ hệ thống src
from src.models.cnn_model import CNN1DModel
from src.models.mlp_model import MLPModel
from src.training.trainer import NNTrainer  # Hàm loss custom dạng PyTorch Tensor

In [6]:
FEATURES_PATH = '/kaggle/input/notebooks/thnhvinhs/notebook22d94af6ba/Realized-Volatility-Prediction/data/processed/features_FINAL.feather'

print("📊 Đang nạp tập dữ liệu đặc trưng toàn cục...")
df = pd.read_feather(FEATURES_PATH)
print(f"Kích thước ma trận dữ liệu: {df.shape}")

# 1. Khôi phục trục dòng thời gian thực tế
splitter = TimeSeriesCVSplitter(n_splits=4, random_state=42)
if 'true_time_id' not in df.columns:
    print("⏳ Đang tính toán t-SNE để tái cấu trúc trục thời gian...")
    time_order = splitter.reverse_engineer_time_order(df, df)
    df = df.merge(time_order, on='time_id', how='left')

# 2. Phân định danh sách đặc trưng học tập
not_features = ['row_id', 'time_id', 'stock_id', 'target', 'true_time_id', 'tsne_val']
features = [col for col in df.columns if col not in not_features]

print(f"🛠️ Đang xử lý làm sạch dữ liệu chuyên sâu cho Mạng Nơ-ron...")
# Xử lý các giá trị vô cùng (nếu có) sinh ra từ các phép chia đặc trưng vi cấu trúc sổ lệnh
df[features] = df[features].replace([np.inf, -np.inf], np.nan)

# Điền các giá trị trống bằng giá trị trung bình của từng cột đặc trưng tương ứng
# Sử dụng phương pháp gán giá trị nhanh, tối ưu hóa vùng nhớ
df[features] = df[features].fillna(df[features].mean())

print(f"✅ Dữ liệu đã được làm sạch 100%. Sẵn sàng đưa vào không gian Tensor.")

📊 Đang nạp tập dữ liệu đặc trưng toàn cục...
Kích thước ma trận dữ liệu: (428932, 574)
⏳ Đang tính toán t-SNE để tái cấu trúc trục thời gian...
2026-06-09 04:44:19 | INFO     | cv_splitter | Đang khôi phục trật tự thời gian bằng t-SNE...
2026-06-09 04:50:14 | INFO     | cv_splitter | Hoàn tất khôi phục trật tự thời gian.
🛠️ Đang xử lý làm sạch dữ liệu chuyên sâu cho Mạng Nơ-ron...
✅ Dữ liệu đã được làm sạch 100%. Sẵn sàng đưa vào không gian Tensor.


In [7]:
# Thuật toán lọc mô hình của SoTA
def get_top_n_models(model_records, top_n):
    if len(model_records) <= top_n:
        return model_records
    # Sắp xếp theo điểm số (loss) từ thấp đến cao
    sorted_records = sorted(model_records, key=lambda x: x['score'])
    print(f"Danh sách điểm số sau khi lọc: {[x['score'] for x in sorted_records]}")
    return sorted_records[:top_n]

In [8]:
EPOCHS = 40
BATCH_SIZE_CNN = 1280  # Tác giả dùng batch size lớn cho CNN
BATCH_SIZE_MLP = 512   # Tác giả dùng batch size nhỏ cho MLP để tăng noise, chống overfitting
NN_NUM_MODELS = 3      # Số lượng Seed thử nghiệm mỗi Fold (Tăng lên 5 nếu có thời gian)
NN_MODEL_TOP_N = 2     # Chỉ giữ lại 2 Seed tốt nhất

# Ngưỡng an toàn: Nếu OOF Loss > ngưỡng này, coi như mô hình bị "tẩu hỏa nhập ma"
CNN_VALID_TH = 0.35
MLP_VALID_TH = 0.30

cnn_oof_preds = np.zeros(len(df))
mlp_oof_preds = np.zeros(len(df))

os.makedirs('./models/cnn', exist_ok=True)
os.makedirs('./models/mlp', exist_ok=True)

input_dim = len(features)
folds_generator = splitter.split(df)

for fold, (train_idx, val_idx) in enumerate(folds_generator):
    print(f"\n" + "═"*50 + f" 🧪 THỰC NGHIỆM SO SÁNH FOLD {fold + 1}/4 " + "═"*50)
    
    X_train_raw, y_train_raw = df.loc[train_idx, features].values, df.loc[train_idx, 'target'].values
    X_val_raw, y_val_raw = df.loc[val_idx, features].values, df.loc[val_idx, 'target'].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    
    # --- THỰC NGHIỆM 1D-CNN ---
    print("\n   ↳ 🎥 [1D-CNN] Bắt đầu chiến thuật Multi-Seed...")
    cnn_fold_records = []
    
    for seed in range(NN_NUM_MODELS):
        print(f"      ▶️ Đang huấn luyện CNN - Seed {seed}...")
        torch.manual_seed(seed) # Khởi tạo trọng số ngẫu nhiên theo seed
        
        cnn_model = CNN1DModel(num_features=input_dim)
        cnn_trainer = NNTrainer(model=cnn_model, device='cuda', lr=0.0011, weight_decay=6.5e-6)
        
        score = cnn_trainer.train(
            X_train_scaled, y_train_raw, X_val_scaled, y_val_raw, 
            batch_size=BATCH_SIZE_CNN, epochs=EPOCHS, patience=8 # Tăng patience theo SoTA
        )
        
        if score < CNN_VALID_TH:
            print(f"      ✅ Seed {seed} ĐẠT YÊU CẦU (Loss: {score:.5f}). Đưa vào danh sách chờ.")
            preds = cnn_trainer.predict(X_val_scaled, batch_size=BATCH_SIZE_CNN)
            cnn_fold_records.append({'score': score, 'preds': preds, 'seed': seed})
        else:
            print(f"      ❌ Seed {seed} PHÂN KỲ (Loss: {score:.5f} > {CNN_VALID_TH}). LOẠI BỎ!")
            
    # Lọc lại Top N CNN tốt nhất
    top_cnn = get_top_n_models(cnn_fold_records, NN_MODEL_TOP_N)
    if top_cnn:
        # Lấy trung bình cộng của các mảng dự đoán (preds) từ các mô hình top
        cnn_oof_preds[val_idx] = np.mean([x['preds'] for x in top_cnn], axis=0)
    
    # --- THỰC NGHIỆM MLP ---
    print("\n   ↳ ⚡ [MLP] Bắt đầu chiến thuật Multi-Seed...")
    mlp_fold_records = []
    
    for seed in range(NN_NUM_MODELS):
        print(f"      ▶️ Đang huấn luyện MLP - Seed {seed}...")
        torch.manual_seed(seed)
        
        mlp_model = MLPModel(num_features=input_dim)
        mlp_trainer = NNTrainer(model=mlp_model, device='cuda', lr=0.002, weight_decay=1e-7)
        
        score = mlp_trainer.train(
            X_train_scaled, y_train_raw, X_val_scaled, y_val_raw, 
            batch_size=BATCH_SIZE_MLP, epochs=EPOCHS, patience=8
        )
        
        if score < MLP_VALID_TH:
            print(f"      ✅ Seed {seed} ĐẠT YÊU CẦU (Loss: {score:.5f}). Đưa vào danh sách chờ.")
            preds = mlp_trainer.predict(X_val_scaled, batch_size=BATCH_SIZE_MLP)
            mlp_fold_records.append({'score': score, 'preds': preds, 'seed': seed})
        else:
            print(f"      ❌ Seed {seed} PHÂN KỲ (Loss: {score:.5f} > {MLP_VALID_TH}). LOẠI BỎ!")
            
    # Lọc lại Top N MLP tốt nhất
    top_mlp = get_top_n_models(mlp_fold_records, NN_MODEL_TOP_N)
    if top_mlp:
        mlp_oof_preds[val_idx] = np.mean([x['preds'] for x in top_mlp], axis=0)
        
    del X_train_scaled, X_val_scaled, cnn_model, mlp_model
    gc.collect()


══════════════════════════════════════════════════ 🧪 THỰC NGHIỆM SO SÁNH FOLD 1/4 ══════════════════════════════════════════════════

   ↳ 🎥 [1D-CNN] Bắt đầu chiến thuật Multi-Seed...
      ▶️ Đang huấn luyện CNN - Seed 0...
2026-06-09 04:50:40 | INFO     | nn_trainer | Bắt đầu Training. Thiết bị: cuda | Epochs: 40 | Batch size: 1280
2026-06-09 04:51:09 | INFO     | nn_trainer | Epoch 001 | Train Loss: 47.74074 | Val Loss: 5.12047
2026-06-09 04:52:03 | INFO     | nn_trainer | Epoch 003 | Train Loss: 6.53033 | Val Loss: 2.10630
2026-06-09 04:53:00 | INFO     | nn_trainer | Epoch 005 | Train Loss: 5.41441 | Val Loss: 1.69879
2026-06-09 04:54:55 | INFO     | nn_trainer | Epoch 009 | Train Loss: 4.19259 | Val Loss: 1.25452
2026-06-09 04:55:53 | INFO     | nn_trainer | Epoch 011 | Train Loss: 3.47813 | Val Loss: 2.45403
2026-06-09 04:56:51 | INFO     | nn_trainer | Epoch 013 | Train Loss: 7.42556 | Val Loss: 1.13844
2026-06-09 05:00:40 | INFO     | nn_trainer | Epoch 021 | Train Loss: 3.15

In [9]:
valid_indexes_cnn = cnn_oof_preds > 0
if np.sum(valid_indexes_cnn) > 0:
    final_cnn_score = rmspe(df.loc[valid_indexes_cnn, 'target'].values, cnn_oof_preds[valid_indexes_cnn])
    print(f"🏆 [SO SÁNH THỰC NGHIỆM] ĐIỂM SỐ MỚI CỦA 1D-CNN: {final_cnn_score:.5f}")
else:
    print("⚠️ Toàn bộ các mô hình CNN đều bị loại bỏ do không đạt ngưỡng.")

valid_indexes_mlp = mlp_oof_preds > 0
if np.sum(valid_indexes_mlp) > 0:
    final_mlp_score = rmspe(df.loc[valid_indexes_mlp, 'target'].values, mlp_oof_preds[valid_indexes_mlp])
    print(f"🏆 [SO SÁNH THỰC NGHIỆM] ĐIỂM SỐ MỚI CỦA MLP   : {final_mlp_score:.5f}")

🏆 [SO SÁNH THỰC NGHIỆM] ĐIỂM SỐ MỚI CỦA 1D-CNN: 0.26506
🏆 [SO SÁNH THỰC NGHIỆM] ĐIỂM SỐ MỚI CỦA MLP   : 0.22819
